In [2]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install -q -U google-genai chromadb sentence-transformers

In [4]:
import os
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from google import genai

In [5]:
RAG_DATASET = "../Data/rag/rico_rag_dataset.csv"

VECTOR_DB = "../Data/vector_db"

print("RAG dataset:", RAG_DATASET)
print("Vector DB:", VECTOR_DB)

RAG dataset: ../Data/rag/rico_rag_dataset.csv
Vector DB: ../Data/vector_db


In [6]:
rag_df = pd.read_csv(RAG_DATASET)

print("Dataset shape:", rag_df.shape)

rag_df.head()

Dataset shape: (29, 5)


,record_id,record_type,title,text,source
0,RAG_00000,component_profile,UI Component: Button,The UI component type Button appears 41 times ...,structured_component_dataset_v1_clustered.csv
1,RAG_00001,component_profile,UI Component: CheckBox,The UI component type CheckBox appears 5 times...,structured_component_dataset_v1_clustered.csv
2,RAG_00002,component_profile,UI Component: CheckedTextView,The UI component type CheckedTextView appears ...,structured_component_dataset_v1_clustered.csv
3,RAG_00003,component_profile,UI Component: EditText,The UI component type EditText appears 38 time...,structured_component_dataset_v1_clustered.csv
4,RAG_00004,cluster_profile,UI Component Cluster 0,Cluster 0 contains 128 UI components. The most...,structured_component_dataset_v1_clustered.csv


In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [8]:
documents = rag_df["text"].fillna("").tolist()

embeddings = embedding_model.encode(documents, show_progress_bar=True)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (29, 384)


In [9]:
client = chromadb.PersistentClient(path=VECTOR_DB)

collection = client.get_or_create_collection(name="rico_ui_knowledge")

print("Existing documents:", collection.count())

Existing documents: 29


In [10]:
ids = rag_df["record_id"].astype(str).tolist()

metadatas = (
    rag_df[["record_type", "title", "source"]].fillna("").to_dict(orient="records")
)

collection.upsert(
    ids=ids, documents=documents, embeddings=embeddings.tolist(), metadatas=metadatas
)

print("Documents in ChromaDB:", collection.count())

Documents in ChromaDB: 29


In [11]:
query = "What components commonly appear with a Button?"

query_embedding = embedding_model.encode([query])[0]

results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)

for i, document in enumerate(results["documents"][0]):

    print(f"\n--- Result {i+1} ---")
    print(document)


--- Result 1 ---
The UI component type Button appears 41 times in the analyzed RICO screenshots. Its average width is 82.84 pixels, average height is 197.30 pixels, average area is 16351.74 square pixels, and average aspect ratio is 0.42. The average detection confidence is 0.830.

--- Result 2 ---
Button and CheckBox occur together in 5 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.

--- Result 3 ---
Button and EditText occur together in 37 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.

--- Result 4 ---
Button and CheckedTextView occur together in 18 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.

--- Result 5 ---
A Button is positioned below a EditText in the analyzed UI screenshots. This spatial relationship was observed 1 times.


In [12]:
def retrieve_context(query, top_k=5):

    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()], n_results=top_k
    )

    documents = results["documents"][0]

    context = "\n\n".join([f"[Source {i+1}]\n{doc}" for i, doc in enumerate(documents)])

    return context

In [13]:
context = retrieve_context(
    "What are the common spatial relationships involving EditText?"
)

print(context)

[Source 1]
A EditText is positioned right of a CheckBox in the analyzed UI screenshots. This spatial relationship was observed 5 times.

[Source 2]
A EditText is positioned right of a CheckedTextView in the analyzed UI screenshots. This spatial relationship was observed 10 times.

[Source 3]
A Button is positioned below a EditText in the analyzed UI screenshots. This spatial relationship was observed 1 times.

[Source 4]
A Button is positioned left of a EditText in the analyzed UI screenshots. This spatial relationship was observed 3 times.

[Source 5]
A Button is positioned above a EditText in the analyzed UI screenshots. This spatial relationship was observed 33 times.


In [14]:
!pip install -U python-dotenv

In [15]:
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [16]:
gemini_client = genai.Client(api_key=api_key)

print("Gemini ready")

Gemini ready


In [17]:
def build_prompt(question, context):

    prompt = f"""
You are an AI assistant for understanding
Android user interfaces using observations
from the RICO dataset.

Answer the user's question using ONLY
the provided context.

Do not invent information.

If the context does not contain enough
information to answer the question,
say that the information was not found
in the analyzed RICO data.

The observations describe the analyzed
RICO sample and should not automatically
be treated as universal Android UI rules.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    return prompt

In [18]:
def ask_rico(question, top_k=5):

    context = retrieve_context(question, top_k)

    prompt = build_prompt(question, context)

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash", contents=prompt
    )

    return response.text

In [19]:
answer = ask_rico("What components commonly appear with a Button?")

print(answer)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Based on the analyzed RICO data:

Components that commonly appear with a Button are:
*   EditText (occurring together in 37 analyzed screenshots)
*   CheckedTextView (occurring together in 18 analyzed screenshots)
*   CheckBox (occurring together in 5 analyzed screenshots)


In [20]:
print(ask_rico("What are the common spatial relationships involving EditText?"))

The common spatial relationships involving EditText in the analyzed RICO data are:

*   A EditText is positioned right of a CheckBox (5 times).
*   A EditText is positioned right of a CheckedTextView (10 times).
*   A Button is positioned below a EditText (1 time).
*   A Button is positioned left of a EditText (3 times).
*   A Button is positioned above a EditText (33 times).


In [21]:
print(ask_rico("What UI patterns were observed in the RICO sample?"))

The following UI patterns were observed in the analyzed RICO sample:
*   Button and CheckBox occur together in 5 analyzed screenshots.
*   Button and EditText occur together in 37 analyzed screenshots.
*   CheckBox and EditText occur together in 5 analyzed screenshots.
*   Button and CheckedTextView occur together in 18 analyzed screenshots.


In [22]:
def retrieve_with_sources(query, top_k=5):

    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()], n_results=top_k
    )

    return (results["documents"][0], results["metadatas"][0])

In [23]:
sources, metadata = retrieve_with_sources(
    "What components commonly appear with a Button?"
)

for i, source in enumerate(sources):

    print(f"\n--- Retrieved Record {i+1} ---")

    print(source)

    print("Metadata:", metadata[i])


--- Retrieved Record 1 ---
The UI component type Button appears 41 times in the analyzed RICO screenshots. Its average width is 82.84 pixels, average height is 197.30 pixels, average area is 16351.74 square pixels, and average aspect ratio is 0.42. The average detection confidence is 0.830.
Metadata: {'title': 'UI Component: Button', 'source': 'structured_component_dataset_v1_clustered.csv', 'record_type': 'component_profile'}

--- Retrieved Record 2 ---
Button and CheckBox occur together in 5 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.
Metadata: {'record_type': 'component_cooccurrence', 'title': 'Button and CheckBox Co-occurrence', 'source': 'component_cooccurrence.csv'}

--- Retrieved Record 3 ---
Button and EditText occur together in 37 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.
Metadata: {'record_type': 'component_cooccurrence', 'source': 'component_cooccurrence.csv',

In [34]:
print(ask_rico("What is the most common color ?"))

The information was not found in the analyzed RICO data.
